# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [ ]:
import glob
import os
import wer
import observation_model
import openfst_python as fst
from decoder import MyViterbiDecoder
from utils import parse_lexicon, generate_symbol_tables
import math
import time
from collections import Counter, defaultdict


def compute_unigram_probs(wav_files):
    counts = Counter()
    for wav_file in wav_files:
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f:
            words = f.readline().strip().split()
        counts.update(words)
    total = sum(counts.values())
    return {word: count / total for word, count in counts.items()}


def compute_bigram_probs(wav_files, vocab, smoothing=1):
    bigram_counts = defaultdict(Counter)
    unigram_counts = Counter()

    for wav_file in wav_files:
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f:
            words = f.readline().strip().split()
        for i in range(len(words) - 1):
            bigram_counts[words[i]][words[i + 1]] += 1
            unigram_counts[words[i]] += 1
        if words:
            unigram_counts[words[-1]] += 1

    V = len(vocab)
    bigram_probs = {}
    for w1 in vocab:
        bigram_probs[w1] = {}
        for w2 in vocab:
            count = bigram_counts[w1][w2] + smoothing
            total = unigram_counts[w1] + smoothing * V
            bigram_probs[w1][w2] = count / total

    return bigram_probs


def compute_interpolated_probs(unigram_probs, bigram_probs, vocab, lam=1.0):
    interpolated = {}
    for w1 in vocab:
        interpolated[w1] = {}
        for w2 in vocab:
            p_bigram = bigram_probs[w1][w2]
            p_unigram = unigram_probs.get(w2, 1.0 / len(vocab))
            interpolated[w1][w2] = lam * p_bigram + (1.0 - lam) * p_unigram
    return interpolated


def ensure_symbol(table, symbol):
    symbol_id = table.find(symbol)
    if symbol_id == -1:
        symbol_id = table.add_symbol(symbol)
    return symbol_id


def ensure_disambig_hmm_states(state_table, n_states=3, phone_symbol='#0'):
    """
    Lexicon disambiguation uses a fake phone (e.g. '#0') on the *phone* tape. For H ∘ L to
    compose, H must emit that phone from real HMM-state input labels. Add #0_1..#0_n to
    state_table if missing (same layout as normal phones). The NN may treat these as OOV
    states; that is still preferable to omitting #0 from H, which breaks composition.
    """
    for i in range(1, n_states + 1):
        sym = f'{phone_symbol}_{i}'
        if state_table.find(sym) == -1:
            state_table.add_symbol(sym)


def build_L(lexicon, phone_table, word_table, add_disambig=True):
    """Lexicon transducer: phones -> words, with closure and optional #0 disambiguation."""
    L = fst.Fst('log')

    disambig_phone_id = None
    if add_disambig:
        disambig_phone_id = ensure_symbol(phone_table, '#0')
        # Word-side #0 keeps symbol tables aligned if tools inspect output alphabets.
        ensure_symbol(word_table, '#0')

    # Attach symbol tables after possible disambiguation updates.
    L.set_input_symbols(phone_table)
    L.set_output_symbols(word_table)

    start = L.add_state()
    L.set_start(start)
    # Do NOT mark start as final: that would allow an empty utterance (zero words) with
    # finite cost and can inflate insertions in WER. Only end-of-word states are finals.
    for word, phones in lexicon.items():
        current = start
        word_id = word_table.find(word)

        for i, phone in enumerate(phones):
            phone_id = phone_table.find(phone)
            next_state = L.add_state()
            out_label = word_id if i == len(phones) - 1 else 0
            L.add_arc(current, fst.Arc(phone_id, out_label, fst.Weight.One('log'), next_state))
            current = next_state

        if add_disambig:
            dis_state = L.add_state()
            L.add_arc(current, fst.Arc(disambig_phone_id, 0, fst.Weight.One('log'), dis_state))
            current = dis_state

        L.add_arc(current, fst.Arc(0, 0, fst.Weight.One('log'), start))
        L.set_final(current)

    return L


def build_G(vocab, word_table, bigram_probs, unigram_probs, lam=1.0):
    """Grammar transducer: words -> words with interpolated bigram LM weights."""
    G = fst.Fst('log')
    G.set_input_symbols(word_table)
    G.set_output_symbols(word_table)

    interpolated = compute_interpolated_probs(unigram_probs, bigram_probs, vocab, lam=lam)

    start = G.add_state()
    G.set_start(start)

    word_state = {}
    for w in vocab:
        s = G.add_state()
        word_state[w] = s
        G.set_final(s)

    for w in vocab:
        wid = word_table.find(w)
        p = max(unigram_probs.get(w, 1.0 / len(vocab)), 1e-12)
        G.add_arc(start, fst.Arc(wid, wid, fst.Weight('log', -math.log(p)), word_state[w]))

    for w1 in vocab:
        s1 = word_state[w1]
        for w2 in vocab:
            wid2 = word_table.find(w2)
            p = max(interpolated[w1][w2], 1e-12)
            G.add_arc(s1, fst.Arc(wid2, wid2, fst.Weight('log', -math.log(p)), word_state[w2]))

    return G


def build_H(phone_table, state_table, n_states=3, stay_cost=0.9):
    """Acoustic model transducer: HMM-state labels -> phone labels (incl. lexicon #0 if present)."""
    H = fst.Fst('log')
    H.set_input_symbols(state_table)
    H.set_output_symbols(phone_table)

    trans_cost = 1.0 - stay_cost
    start = H.add_state()
    H.set_start(start)
    H.set_final(start)

    # Only skip epsilon; '#0' must be built if state_table has #0_1..#0_n (see ensure_disambig_hmm_states).
    skip_phones = {'<eps>'}

    # Iterate by symbol IDs to stay compatible with openfst-python table API.
    for phone_id in range(phone_table.num_symbols()):
        p = phone_table.find(phone_id)
        if p in skip_phones:
            continue

        current = start

        for i in range(1, n_states + 1):
            state_id = state_table.find(f"{p}_{i}")
            if state_id == -1:
                continue

            H.add_arc(current, fst.Arc(state_id, 0, fst.Weight('log', -math.log(stay_cost)), current))

            next_state = H.add_state()
            out_label = phone_id if i == n_states else 0
            H.add_arc(current, fst.Arc(state_id, out_label, fst.Weight('log', -math.log(trans_cost)), next_state))
            current = next_state

        H.add_arc(current, fst.Arc(0, 0, fst.Weight.One('log'), start))

    return H


def push_weights_safe(fst_obj, label='FST', verbose=True):
    """
    LM look-ahead: push weights toward the input (phones). If push fails, we keep the
    unstopped graph and log so experiments are not silently interpreted as 'pushed'.
    """
    try:
        fst_obj.push(push_weights=True, to_final=False)
        if verbose:
            print(f"[push] OK: {label}")
        return fst_obj
    except Exception as e1:
        try:
            out = fst.push(fst_obj, push_weights=True, to_final=False)
            if verbose:
                print(f"[push] OK (fst.push): {label}")
            return out
        except Exception as e2:
            if verbose:
                print(f"[push] FAILED {label}: {e1!r}; fallback {e2!r} — continuing without push.")
            return fst_obj


def minimize_safe(fst_obj, label='FST', verbose=True):
    try:
        fst_obj.minimize()
        if verbose:
            print(f"[minimize] OK: {label}")
        return fst_obj
    except Exception as e:
        if verbose:
            print(f"[minimize] skipped {label}: {e!r}")
        return fst_obj


def build_hlg(lexicon, vocab, word_table, phone_table, state_table, bigram_probs, unigram_probs, lam=1.0, n_states=3, stay_cost=0.9, verbose_graph_ops=True):
    L = build_L(lexicon, phone_table, word_table, add_disambig=True)
    G = build_G(vocab, word_table, bigram_probs, unigram_probs, lam=lam)

    # After L is built, '#0' exists on the phone tape — ensure HMM ids exist before build_H.
    ensure_disambig_hmm_states(state_table, n_states=n_states, phone_symbol='#0')

    L.arcsort(sort_type='olabel')
    G.arcsort(sort_type='ilabel')

    LG = fst.compose(L, G)
    # Remove unreachable states so determinize/push see a single connected component.
    LG = fst.connect(LG)
    LG_det = fst.determinize(LG)
    LG_push = push_weights_safe(LG_det, label='LG after determinize', verbose=verbose_graph_ops)
    LG_opt = minimize_safe(LG_push, label='LG after push', verbose=verbose_graph_ops)

    H = build_H(phone_table, state_table, n_states=n_states, stay_cost=stay_cost)
    H.arcsort(sort_type='olabel')
    LG_opt.arcsort(sort_type='ilabel')

    HLG = fst.compose(H, LG_opt)
    HLG = fst.connect(HLG)
    HLG.set_input_symbols(state_table)
    HLG.set_output_symbols(word_table)
    return HLG


def create_wfst_monolithic(lexicon, word_table, phone_table, state_table, interpolated_probs, unigram_probs, n_states=3, stay_cost=0.9):
    """Baseline: previous single-graph implementation (without silence branch)."""
    f = fst.Fst('log')
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)

    trans_cost = 1.0 - stay_cost
    start_state = f.add_state()
    f.set_start(start_state)

    word_loop_states = {}
    for word in lexicon:
        word_loop_states[word] = f.add_state()
        f.set_final(word_loop_states[word])

    for word in lexicon:
        p_unigram = max(unigram_probs.get(word, 1.0 / len(lexicon)), 1e-12)
        f.add_arc(start_state, fst.Arc(0, 0, fst.Weight('log', -math.log(p_unigram)), word_loop_states[word]))

    for word, phones in lexicon.items():
        word_id = word_table.find(word)

        for next_word in lexicon:
            p_interp = max(interpolated_probs[word][next_word], 1e-12)
            f.add_arc(word_loop_states[word], fst.Arc(0, 0, fst.Weight('log', -math.log(p_interp)), word_loop_states[next_word]))

        current_state = word_loop_states[word]

        for phone_idx, phone in enumerate(phones):
            for i in range(1, n_states + 1):
                in_label = state_table.find(f"{phone}_{i}")
                f.add_arc(current_state, fst.Arc(in_label, 0, fst.Weight('log', -math.log(stay_cost)), current_state))

                next_state = f.add_state()
                out_label = word_id if (phone_idx == len(phones) - 1 and i == n_states) else 0
                f.add_arc(current_state, fst.Arc(in_label, out_label, fst.Weight('log', -math.log(trans_cost)), next_state))
                current_state = next_state

        f.add_arc(current_state, fst.Arc(0, 0, fst.Weight.One('log'), word_loop_states[word]))

    return f


def run_decode(f, wav_files, beam=100):
    total_substitutions = 0
    total_deletions = 0
    total_insertions = 0
    total_words = 0
    total_forward_computations = 0
    total_decode_time = 0.0
    total_backtrace_time = 0.0

    for wav_file in wav_files:
        decoder = MyViterbiDecoder(f, wav_file, beam=beam)

        t0 = time.perf_counter()
        decoder.decode()
        decode_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        (state_path, words) = decoder.backtrace()
        backtrace_time = time.perf_counter() - t0

        total_decode_time += decode_time
        total_backtrace_time += backtrace_time
        total_forward_computations += decoder.forward_computation_count

        words_str = ' '.join(words)
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f_txt:
            transcription = f_txt.readline().strip()
        error_counts = wer.compute_alignment_errors(transcription, words_str)
        word_count = len(transcription.split())

        total_substitutions += error_counts[0]
        total_deletions += error_counts[1]
        total_insertions += error_counts[2]
        total_words += word_count

    overall_wer = (total_substitutions + total_deletions + total_insertions) / total_words
    return {
        'wer': overall_wer,
        'sub': total_substitutions,
        'del': total_deletions,
        'ins': total_insertions,
        'fwd': total_forward_computations,
        'decode_time': total_decode_time,
        'backtrace_time': total_backtrace_time
    }


def print_results(label, r, num_states, num_arcs):
    print(f"=== {label} ===")
    print(f"  WER                        : {r['wer']:.2%}")
    print(f"  Total substitutions        : {r['sub']}")
    print(f"  Total deletions            : {r['del']}")
    print(f"  Total insertions           : {r['ins']}")
    print(f"  Total forward computations : {r['fwd']}")
    print(f"  Total decode time          : {r['decode_time']:.4f}s")
    print(f"  Total backtrace time       : {r['backtrace_time']:.4f}s")
    print(f"  Total time                 : {r['decode_time'] + r['backtrace_time']:.4f}s")
    print(f"  WFST states                : {num_states}")
    print(f"  WFST arcs                  : {num_arcs}\n")


lex = parse_lexicon("lexicon.txt")
word_table, phone_table, state_table = generate_symbol_tables(lex)

beam = 50
stay_cost = 0.9
lam = 1.0

wav_files = glob.glob('/group/teaching/asr/labs/recordings/*.wav')
vocab = list(lex.keys())

unigram_probs = compute_unigram_probs(wav_files)
bigram_probs = compute_bigram_probs(wav_files, vocab)
interpolated_probs = compute_interpolated_probs(unigram_probs, bigram_probs, vocab, lam=lam)

baseline_wfst = create_wfst_monolithic(
    lex, word_table, phone_table, state_table,
    interpolated_probs, unigram_probs, stay_cost=stay_cost
)

hlg_wfst = build_hlg(
    lex, vocab, word_table, phone_table, state_table,
    bigram_probs, unigram_probs, lam=lam, stay_cost=stay_cost
)

for label, graph in [
    ("Monolithic interpolated WFST", baseline_wfst),
    ("Tree lexicon + LM look-ahead (H o L o G)", hlg_wfst),
]:
    num_states = graph.num_states()
    num_arcs = sum(1 for s in graph.states() for _ in graph.arcs(s))
    result = run_decode(graph, wav_files, beam=beam)
    print_results(label, result, num_states, num_arcs)

=== Interpolated LM (lambda=0.0) ===
  WER                        : 48.05%
  Total substitutions        : 389
  Total deletions            : 279
  Total insertions           : 132
  Total forward computations : 10683986
  Total decode time          : 179.0554s
  Total backtrace time       : 0.0615s
  Total time                 : 179.1169s
  WFST states                : 121
  WFST arcs                  : 351

=== Interpolated LM (lambda=0.2) ===
  WER                        : 47.93%
  Total substitutions        : 392
  Total deletions            : 275
  Total insertions           : 131
  Total forward computations : 10672966
  Total decode time          : 179.3740s
  Total backtrace time       : 0.0624s
  Total time                 : 179.4364s
  WFST states                : 121
  WFST arcs                  : 351

=== Interpolated LM (lambda=0.4) ===
  WER                        : 47.81%
  Total substitutions        : 391
  Total deletions            : 275
  Total insertions           : 